# Experiment 10 - Deep Learning Case Studies Across Domains

**Aim:** Analyse how CNNs, RNNs, and GANs are applied in healthcare, natural
language processing, autonomous vehicles, finance, and art, and connect the
case studies to small hands-on demonstrations of each architecture family.

**Case-study objectives:**

1. Evaluate the effectiveness of different deep learning architectures in
   different domains.
2. Identify their strengths and weaknesses.
3. Ground the discussion in real-world examples and reproducible code.

**Important note:** the original lab sheet is a written case study. This
notebook preserves that analysis and adds a compact, runnable demonstration of
the three architecture families. The demos are intentionally small so they run
on a free Colab GPU; they are illustrations, not reproductions of the published
systems described in the case studies.

**Datasets:**

- MNIST for the CNN and GAN demos:
  `https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz`
- IMDB for the RNN demo:
  `https://storage.googleapis.com/tensorflow/tf-keras-datasets/imdb.npz`

No local dataset is used.

**Runtime:** about 5-10 minutes on a GPU; longer on CPU.

## 1. Case-study overview

### Convolutional Neural Networks (CNNs)

CNNs process grid-structured data. Convolutional filters learn local patterns,
pooling reduces resolution, and fully connected layers classify the learned
features. They are the standard architecture for image recognition, medical
imaging, and video analysis.

**Healthcare example:** Esteva et al. (2017) trained a CNN on more than
130,000 dermatology images for skin-cancer classification. The model matched
board-certified dermatologists on the reported task, reducing false negatives
and speeding up image review. Challenges include patient privacy,
demographic generalization, and regulatory approval.

### Recurrent Neural Networks (RNNs)

RNNs maintain a hidden state across time, which makes them suitable for text,
speech, and time series. LSTM and GRU variants mitigate vanishing gradients.
Transformers have largely replaced RNNs for long text, but RNNs remain useful
for small sequence models and streaming data.

**NLP example:** an LSTM with attention over more than one million tweets
classified sentiment into positive, negative, and neutral categories with
about 88% accuracy. Challenges include sarcasm, class imbalance, and domain
shift.

**Finance example:** an RNN trained on historical prices and news sentiment
outperformed traditional statistical baselines for short-horizon forecasting.
Challenges include market volatility, regime changes, and data quality.

### Generative Adversarial Networks (GANs)

A GAN pits a generator against a discriminator. The generator turns random
noise into synthetic samples; the discriminator learns to distinguish real from
fake. Training is a minimax game and can be unstable.

**Art example:** style-transfer and image-generation systems such as DeepArt
use GAN-like objectives to transform photographs into artistic images.
Challenges include copyright, attribution, and subjective quality control.

### Autonomous vehicles

Tesla's Autopilot combines CNN-based perception with recurrent decision-making
and continuous learning from fleet data. Challenges include long-tail driving
scenarios, data diversity, safety validation, and regulation.

### Cross-cutting challenges

1. **Data requirements:** large labelled datasets are expensive and sometimes
   privacy-sensitive.
2. **Compute:** training deep models needs GPUs or TPUs.
3. **Interpretability:** high-stakes domains need explanations, not only
   predictions.
4. **Overfitting:** regularization, augmentation, and early stopping are
   essential.
5. **Ethics:** bias, consent, accountability, and dual use must be addressed.

## 2. Environment setup

In [ ]:
import os

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ["PYTHONHASHSEED"] = "42"

import json
import random
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")

## 3. Compact datasets

The CNN and GAN demos share the MNIST download. The RNN demo uses a subset of
IMDB so the notebook finishes quickly.

In [ ]:
MNIST_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
IMDB_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/imdb.npz"
IMDB_WORD_INDEX_URL = (
    "https://storage.googleapis.com/tensorflow/tf-keras-datasets/imdb_word_index.json"
)

MNIST_TRAIN = 12000
MNIST_TEST = 2000
IMDB_VOCAB = 5000
MAXLEN = 200
IMDB_TRAIN = 5000
IMDB_TEST = 2000
GAN_TRAIN = 12000
GAN_EPOCHS = 8
GAN_BATCH_SIZE = 128
LATENT_DIM = 100


def load_mnist():
    path = tf.keras.utils.get_file(
        fname="mnist.npz",
        origin=MNIST_URL,
        cache_subdir="datasets",
    )
    with np.load(path) as data:
        return (
            (np.array(data["x_train"]), np.array(data["y_train"])),
            (np.array(data["x_test"]), np.array(data["y_test"])),
        )


(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = load_mnist()

X_train_cnn = (X_train_raw[:MNIST_TRAIN].astype("float32") / 255.0)[..., None]
y_train_cnn = y_train_raw[:MNIST_TRAIN]
X_test_cnn = (X_test_raw[:MNIST_TEST].astype("float32") / 255.0)[..., None]
y_test_cnn = y_test_raw[:MNIST_TEST]

print(f"CNN train: {X_train_cnn.shape}, CNN test: {X_test_cnn.shape}")
print(f"MNIST pixel range: [{X_train_cnn.min():.2f}, {X_train_cnn.max():.2f}]")

assert X_train_cnn.shape[1:] == (28, 28, 1)

## Demo A - CNN for medical-image-style classification

This demo uses MNIST as a stand-in for a 28x28 grayscale medical image. The
architecture is deliberately small: two convolution blocks, max-pooling, and a
dense classifier. The point is to show the full image-classification workflow,
not to claim clinical validity.

In [ ]:
def build_demo_cnn(seed=SEED):
    tf.keras.utils.set_random_seed(seed)
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(28, 28, 1), name="image"),
            tf.keras.layers.Conv2D(16, 3, activation="relu", padding="same", name="conv_1"),
            tf.keras.layers.MaxPooling2D(name="pool_1"),
            tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same", name="conv_2"),
            tf.keras.layers.MaxPooling2D(name="pool_2"),
            tf.keras.layers.Flatten(name="flatten"),
            tf.keras.layers.Dense(64, activation="relu", name="dense_64"),
            tf.keras.layers.Dense(10, activation="softmax", name="digit_probabilities"),
        ],
        name="demo_cnn",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


cnn_model = build_demo_cnn()
cnn_model.summary()

start_time = time.perf_counter()
cnn_history = cnn_model.fit(
    X_train_cnn,
    y_train_cnn,
    epochs=2,
    batch_size=128,
    validation_split=0.1,
    verbose=2,
)
cnn_seconds = time.perf_counter() - start_time

cnn_test_loss, cnn_test_accuracy = cnn_model.evaluate(X_test_cnn, y_test_cnn, verbose=0)
print(f"CNN training time : {cnn_seconds:.2f} seconds")
print(f"CNN test loss     : {cnn_test_loss:.4f}")
print(f"CNN test accuracy : {cnn_test_accuracy:.4f}")

In [ ]:
cnn_probabilities = cnn_model.predict(X_test_cnn, verbose=0)
cnn_predictions = cnn_probabilities.argmax(axis=1)

print("CNN classification report:")
print(
    classification_report(
        y_test_cnn,
        cnn_predictions,
        target_names=[str(digit) for digit in range(10)],
        digits=4,
        zero_division=0,
    )
)

cm = confusion_matrix(y_test_cnn, cnn_predictions, labels=list(range(10)))
fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(cm, cmap="Blues")
ax.set_title("Demo CNN confusion matrix")
ax.set_xlabel("Predicted digit")
ax.set_ylabel("True digit")
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(range(10))
ax.set_yticklabels(range(10))
for row in range(cm.shape[0]):
    for column in range(cm.shape[1]):
        color = "white" if cm[row, column] > cm.max() / 2 else "black"
        ax.text(column, row, str(cm[row, column]), ha="center", va="center", fontsize=8, color=color)
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp10_cnn_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def predict_mnist_cnn(image):
    image = np.asarray(image, dtype=np.float32)
    if image.shape == (28, 28):
        image = image[..., None]
    if image.shape != (28, 28, 1):
        raise ValueError(f"Expected 28x28 or 28x28x1, got {image.shape}")
    if image.max() > 1.0:
        image = image / 255.0
    probabilities = cnn_model.predict(image.reshape(1, 28, 28, 1), verbose=0)[0]
    digit = int(np.argmax(probabilities))
    return digit, float(probabilities[digit]), probabilities


sample_indices = np.arange(8)
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, index in zip(axes.ravel(), sample_indices):
    ax.imshow(X_test_cnn[index, :, :, 0], cmap="gray", vmin=0, vmax=1)
    predicted, confidence, _ = predict_mnist_cnn(X_test_cnn[index])
    true_label = int(y_test_cnn[index])
    ax.set_title(
        f"true {true_label} / pred {predicted}\nconf {confidence:.3f}",
        fontsize=8,
        color="green" if predicted == true_label else "red",
    )
    ax.axis("off")
fig.suptitle("Demo CNN inference", y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp10_cnn_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

## Demo B - RNN sentiment analysis

This is a compact version of the IMDB sentiment experiment. A 5,000-word
vocabulary and a 32-unit LSTM keep the runtime short while preserving the full
workflow: token sequences, embedding, recurrent encoding, binary output, and
raw-text inference.

In [ ]:
tf.keras.utils.get_file("imdb.npz", IMDB_URL, cache_subdir="datasets")
imdb_word_index_path = tf.keras.utils.get_file(
    "imdb_word_index.json",
    IMDB_WORD_INDEX_URL,
    cache_subdir="datasets",
)

(X_train_imdb, y_train_imdb), (X_test_imdb, y_test_imdb) = tf.keras.datasets.imdb.load_data(
    num_words=IMDB_VOCAB
)
X_train_imdb = tf.keras.utils.pad_sequences(
    X_train_imdb, maxlen=MAXLEN, padding="post", truncating="post"
)[:IMDB_TRAIN]
y_train_imdb = y_train_imdb[:IMDB_TRAIN]
X_test_imdb = tf.keras.utils.pad_sequences(
    X_test_imdb, maxlen=MAXLEN, padding="post", truncating="post"
)[:IMDB_TEST]
y_test_imdb = y_test_imdb[:IMDB_TEST]

with open(imdb_word_index_path, "r", encoding="utf-8") as file:
    raw_word_index = json.load(file)
word_index = {
    word: index + 3
    for word, index in raw_word_index.items()
    if index + 3 < IMDB_VOCAB
}
word_index["<PAD>"] = 0
word_index["<START>"] = 1
word_index["<UNK>"] = 2
word_index["<UNUSED>"] = 3

print(f"IMDB train: {X_train_imdb.shape}, test: {X_test_imdb.shape}")
print(f"Training label counts: {np.bincount(y_train_imdb).tolist()}")
print(f"Test label counts    : {np.bincount(y_test_imdb).tolist()}")

assert X_train_imdb.shape[1] == MAXLEN
assert X_test_imdb.shape[1] == MAXLEN

In [ ]:
def build_demo_rnn(seed=SEED):
    tf.keras.utils.set_random_seed(seed)
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(MAXLEN,), name="review_tokens"),
            tf.keras.layers.Embedding(IMDB_VOCAB, 32, name="embedding"),
            tf.keras.layers.LSTM(32, name="lstm"),
            tf.keras.layers.Dense(1, activation="sigmoid", name="sentiment"),
        ],
        name="demo_sentiment_lstm",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


rnn_model = build_demo_rnn()
rnn_model.summary()

start_time = time.perf_counter()
rnn_history = rnn_model.fit(
    X_train_imdb,
    y_train_imdb,
    epochs=2,
    batch_size=128,
    validation_split=0.1,
    verbose=2,
)
rnn_seconds = time.perf_counter() - start_time

rnn_test_loss, rnn_test_accuracy = rnn_model.evaluate(X_test_imdb, y_test_imdb, verbose=0)
print(f"RNN training time : {rnn_seconds:.2f} seconds")
print(f"RNN test loss     : {rnn_test_loss:.4f}")
print(f"RNN test accuracy : {rnn_test_accuracy:.4f}")

In [ ]:
rnn_probabilities = rnn_model.predict(X_test_imdb, verbose=0).ravel()
rnn_predictions = (rnn_probabilities >= 0.5).astype(int)

print("RNN classification report:")
print(
    classification_report(
        y_test_imdb,
        rnn_predictions,
        target_names=["negative", "positive"],
        digits=4,
        zero_division=0,
    )
)

cm = confusion_matrix(y_test_imdb, rnn_predictions)
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(cm, cmap="Blues")
ax.set_title("Demo RNN sentiment confusion matrix")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["negative", "positive"])
ax.set_yticklabels(["negative", "positive"])
for row in range(2):
    for column in range(2):
        color = "white" if cm[row, column] > cm.max() / 2 else "black"
        ax.text(column, row, str(cm[row, column]), ha="center", va="center", color=color)
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp10_rnn_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def encode_review(text):
    tokens = re.findall(r"[a-z0-9']+", text.lower())
    indices = [1]
    for token in tokens:
        index = word_index.get(token, 2)
        indices.append(index if index < IMDB_VOCAB else 2)
    return tf.keras.utils.pad_sequences(
        [indices], maxlen=MAXLEN, padding="post", truncating="post"
    )[0]


def predict_sentiment(text):
    encoded = encode_review(text)
    probability = float(rnn_model.predict(encoded.reshape(1, MAXLEN), verbose=0)[0, 0])
    label = "positive" if probability >= 0.5 else "negative"
    return label, probability


for review in [
    "A brilliant, moving film with wonderful performances.",
    "A dull and predictable movie that I would not recommend.",
    "The acting was fine, but the story felt rushed.",
]:
    label, probability = predict_sentiment(review)
    print(f"[{label:8s}] P(positive)={probability:.4f} | {review}")

## Demo C - GAN for image generation

The GAN is a small DCGAN-style model:

- **Generator:** Dense 7x7x128 -> reshape -> two transposed convolutions ->
  28x28 tanh output.
- **Discriminator:** two strided convolutions -> flatten -> one logit.

The generator and discriminator are trained adversarially with binary
cross-entropy from logits and Adam with a 0.5 first-moment decay, a standard
choice for stable GAN training.

In [ ]:
X_train_gan = (
    (X_train_raw[:GAN_TRAIN].astype("float32") - 127.5) / 127.5
)[..., None]
print(f"GAN training images: {X_train_gan.shape}")
print(f"GAN image range    : [{X_train_gan.min():.2f}, {X_train_gan.max():.2f}]")

gan_dataset = (
    tf.data.Dataset.from_tensor_slices(X_train_gan)
    .shuffle(buffer_size=len(X_train_gan), seed=SEED)
    .batch(GAN_BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)


def build_generator(seed=SEED):
    tf.keras.utils.set_random_seed(seed)
    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(LATENT_DIM,), name="noise"),
            tf.keras.layers.Dense(7 * 7 * 128, use_bias=False, name="dense_7x7x128"),
            tf.keras.layers.BatchNormalization(name="batch_norm_1"),
            tf.keras.layers.LeakyReLU(negative_slope=0.2, name="leaky_relu_1"),
            tf.keras.layers.Reshape((7, 7, 128), name="reshape"),
            tf.keras.layers.Conv2DTranspose(
                64, 5, strides=2, padding="same", use_bias=False, name="deconv_1"
            ),
            tf.keras.layers.BatchNormalization(name="batch_norm_2"),
            tf.keras.layers.LeakyReLU(negative_slope=0.2, name="leaky_relu_2"),
            tf.keras.layers.Conv2DTranspose(
                1, 5, strides=2, padding="same", activation="tanh", name="generated_image"
            ),
        ],
        name="gan_generator",
    )


def build_discriminator(seed=SEED):
    tf.keras.utils.set_random_seed(seed)
    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(28, 28, 1), name="image"),
            tf.keras.layers.Conv2D(64, 5, strides=2, padding="same", name="conv_1"),
            tf.keras.layers.LeakyReLU(negative_slope=0.2, name="leaky_relu_1"),
            tf.keras.layers.Dropout(0.3, name="dropout_1"),
            tf.keras.layers.Conv2D(128, 5, strides=2, padding="same", name="conv_2"),
            tf.keras.layers.LeakyReLU(negative_slope=0.2, name="leaky_relu_2"),
            tf.keras.layers.Dropout(0.3, name="dropout_2"),
            tf.keras.layers.Flatten(name="flatten"),
            tf.keras.layers.Dense(1, name="real_or_fake_logit"),
        ],
        name="gan_discriminator",
    )


generator = build_generator()
discriminator = build_discriminator()
generator.summary()
discriminator.summary()

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)
generator_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)


def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss


def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)


@tf.function
def train_step(images):
    noise = tf.random.normal([GAN_BATCH_SIZE, LATENT_DIM])
    with tf.GradientTape() as generator_tape, tf.GradientTape() as discriminator_tape:
        generated_images = generator(noise, training=True)
        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)
        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)
    generator_gradients = generator_tape.gradient(gen_loss, generator.trainable_variables)
    discriminator_gradients = discriminator_tape.gradient(
        disc_loss, discriminator.trainable_variables
    )
    generator_optimizer.apply_gradients(
        zip(generator_gradients, generator.trainable_variables)
    )
    discriminator_optimizer.apply_gradients(
        zip(discriminator_gradients, discriminator.trainable_variables)
    )
    return gen_loss, disc_loss


def generate_and_save_images(model, epoch, test_input):
    predictions = model(test_input, training=False)
    fig = plt.figure(figsize=(4, 4))
    for index in range(predictions.shape[0]):
        plt.subplot(4, 4, index + 1)
        plt.imshow(predictions[index, :, :, 0] * 0.5 + 0.5, cmap="gray")
        plt.axis("off")
    fig.suptitle(f"GAN generated digits after epoch {epoch}")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"exp10_gan_epoch_{epoch:02d}.png", dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)


test_noise = tf.random.normal([16, LATENT_DIM], seed=SEED)
gan_history = {"generator_loss": [], "discriminator_loss": []}

for epoch in range(GAN_EPOCHS):
    epoch_start = time.perf_counter()
    last_generator_loss = 0.0
    last_discriminator_loss = 0.0
    for image_batch in gan_dataset:
        last_generator_loss, last_discriminator_loss = train_step(image_batch)
    epoch_seconds = time.perf_counter() - epoch_start

    generator_loss_value = float(last_generator_loss)
    discriminator_loss_value = float(last_discriminator_loss)
    gan_history["generator_loss"].append(generator_loss_value)
    gan_history["discriminator_loss"].append(discriminator_loss_value)
    print(
        f"Epoch {epoch + 1:02d}/{GAN_EPOCHS} | "
        f"generator loss {generator_loss_value:.4f} | "
        f"discriminator loss {discriminator_loss_value:.4f} | "
        f"{epoch_seconds:.2f}s"
    )

    if (epoch + 1) % 2 == 0 or epoch == GAN_EPOCHS - 1:
        generate_and_save_images(generator, epoch + 1, test_noise)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(gan_history["generator_loss"], label="Generator", linewidth=2)
ax.plot(gan_history["discriminator_loss"], label="Discriminator", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("GAN adversarial losses")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp10_gan_losses.png", dpi=150, bbox_inches="tight")
plt.show()


def generate_digits(n=16):
    noise = tf.random.normal([n, LATENT_DIM])
    images = generator(noise, training=False).numpy()
    images = images[..., 0] * 0.5 + 0.5
    return images


generated = generate_digits(16)
fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for ax, image in zip(axes.ravel(), generated):
    ax.imshow(image, cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
fig.suptitle("GAN inference: generated digits", y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp10_gan_generated.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Case-study synthesis

| Domain | Architecture | Application | Reported impact | Main challenges |
|---|---|---|---|---|
| Healthcare | CNN | Skin-cancer image classification (Esteva et al., 2017) | Dermatologist-level accuracy on the reported task; fewer false negatives | Patient privacy, demographic generalization, regulation |
| NLP | LSTM + attention, Transformers | Tweet sentiment analysis | About 88% accuracy on positive/negative/neutral classification | Sarcasm, class imbalance, domain shift |
| Autonomous vehicles | CNN + RNN | Perception and path planning | Real-time obstacle detection and continuous fleet learning | Long-tail scenarios, data diversity, safety validation |
| Finance | RNN | Price forecasting with news sentiment | Improved short-horizon forecasts and risk management | Volatility, regime change, data quality |
| Art | GAN | Style transfer and image generation | Accessible creative tools and new artistic workflows | Copyright, attribution, subjective quality |

### What the demos show

- A CNN can learn spatial features and reach high accuracy on a small image
  subset.
- An LSTM can classify variable-length text after embedding and padding.
- A GAN can learn to generate digit-like images from noise, but the output is
  limited by the small training budget.

### What the demos do not show

- Clinical or financial validity. The datasets are toy proxies.
- Production-scale training, distributed systems, or deployment.
- Fairness, privacy, or regulatory compliance analysis.
- The published systems' exact architectures, data, or results.

### Challenges reinforced by the demos

- **Data requirements:** even a small CNN needs thousands of labelled images.
- **Compute:** the GAN is the most expensive demo despite the tiny images.
- **Interpretability:** the CNN's confusion matrix is easier to audit than the
  GAN's internal representation.
- **Instability:** GAN losses do not decrease monotonically, which is normal
  for adversarial training.

## 5. Save demo models and metrics

In [ ]:
cnn_model_path = OUTPUT_DIR / "exp10_demo_cnn.keras"
rnn_model_path = OUTPUT_DIR / "exp10_demo_rnn.keras"
generator_path = OUTPUT_DIR / "exp10_gan_generator.keras"
discriminator_path = OUTPUT_DIR / "exp10_gan_discriminator.keras"

cnn_model.save(cnn_model_path)
rnn_model.save(rnn_model_path)
generator.save(generator_path)
discriminator.save(discriminator_path)

metrics = pd.DataFrame(
    [
        {
            "Demo": "CNN (MNIST proxy)",
            "Test loss": float(cnn_test_loss),
            "Test accuracy": float(cnn_test_accuracy),
            "Training seconds": float(cnn_seconds),
        },
        {
            "Demo": "RNN (IMDB subset)",
            "Test loss": float(rnn_test_loss),
            "Test accuracy": float(rnn_test_accuracy),
            "Training seconds": float(rnn_seconds),
        },
        {
            "Demo": "GAN (MNIST)",
            "Test loss": float(gan_history["discriminator_loss"][-1]),
            "Test accuracy": float("nan"),
            "Training seconds": float("nan"),
        },
    ]
)
metrics.to_csv(OUTPUT_DIR / "exp10_demo_metrics.csv", index=False)

print(f"Saved CNN model         : {cnn_model_path.resolve()}")
print(f"Saved RNN model         : {rnn_model_path.resolve()}")
print(f"Saved GAN generator     : {generator_path.resolve()}")
print(f"Saved GAN discriminator : {discriminator_path.resolve()}")
print(metrics.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

## 6. Conclusion

The case studies show that architecture choice follows the structure of the
data and the task:

- **CNNs** dominate grid-structured perception problems such as medical
  imaging and object recognition.
- **RNNs/LSTMs** handle ordered sequences and remain useful for compact text
  and time-series models, although Transformers now dominate large-scale NLP.
- **GANs** enable data generation and creative tools but are harder to train
  and evaluate.

Across all domains, the recurring constraints are data quality, compute,
interpretability, robustness, and ethics. The hands-on demos make the workflow
concrete: load public data, build a small model, train, evaluate with more than
one metric, and run inference. They are teaching artifacts, not production
systems.

**Artifacts created at runtime (not committed):**

- `outputs/exp10_demo_cnn.keras`
- `outputs/exp10_demo_rnn.keras`
- `outputs/exp10_gan_generator.keras`
- `outputs/exp10_gan_discriminator.keras`
- `outputs/exp10_demo_metrics.csv`
- `outputs/exp10_cnn_confusion_matrix.png`
- `outputs/exp10_cnn_predictions.png`
- `outputs/exp10_rnn_confusion_matrix.png`
- `outputs/exp10_gan_epoch_02.png` ... `outputs/exp10_gan_epoch_08.png`
- `outputs/exp10_gan_losses.png`
- `outputs/exp10_gan_generated.png`

**Execution note:** the notebook was syntax-checked during repository
preparation; the full demo training runs were not performed locally. The
case-study analysis is based on the original lab sheet and its references.

## References

- Esteva, A., et al. "Dermatologist-level classification of skin cancer with
  deep neural networks." *Nature*, 542(7639), 115-118, 2017.
- Malhar and Ram. "A Study on Sentiment Analysis Techniques of Twitter Data."
  *International Journal of Advanced Computer Science and Applications*, 2019.
- Tesla. "Tesla's Autopilot Uses Deep Learning." Tesla Blog.
- DeepArt. "DeepArt Project - Transforming Photos into Art." DeepArt Website.
- "Challenges in Deep Learning Implementation." *Journal of Big Data*.
- Zhang, C., et al. "Understanding Deep Learning Requires Rethinking
  Generalization." ICLR, 2017.
- Lipton, Z. C. "The Mythos of Model Interpretability." *Communications of the
  ACM*, 61(10), 2018.
- Barocas, S., et al. "Big Data's Disparate Impact." *California Law Review*,
  104, 2016.